# Clustering Espacial con HDBSCAN

Un algoritmo de clustering espacial es DBSCAN, que está disponible en ``sklearn.cluster.DBSCAN``.

![](https://upload.wikimedia.org/wikipedia/commons/thumb/a/af/DBSCAN-Illustration.svg/800px-DBSCAN-Illustration.svg.png)

Su nombre significa "barrido basado en densidad." Lo que hace DBSCAN es definir tres tipos de puntos en función de los otros puntos que lo rodean:

- _Núcleos_ (puntos rojos): conjuntos de puntos que, entre sí, están dentro de un radio de distancia y, en total, son más que una cierta cantidad especificada como hiperparámetro.
- _Puntos Alcanzables_ (puntos amarillos): puntos que no son núcleos, pero que están dentro del umbral de tolerancia de distancia a puntos núcleos.
- _Ruido_ (punto azul): puntos que no son alcanzables desde los núcleos.

Considerando eso, se vuelve intuitivo el hecho de que no necesitamos especificar el número de clusters, sino el umbral de distancia para la densidad, y, por tanto, que la forma de los clusters puede ser arbitraria. Al mismo tiempo, habrá puntos que no estarán agrupados en clusters.

Veremos un ejemplo de aplicar una versión de DBSCAN que está en el estado del arte, conocida como [HDBSCAN](https://hdbscan.readthedocs.io/en/latest/) (por Hierarchical DBSCAN). Para instalarla debemos utilizar el siguiente comando:

`pip install hdbscan`

Su API es la misma de Scikit-Learn, así que será sencillo familiarizarnos con su uso.

La diferencia entre HDBSCAN y DBSCAN es que HDBSCAN tiene una noción de distancia umbral flexible. Esto es útil cuando la densidad de los puntos varía y, por tanto, un único umbral podría llevar a resultados erróneos.

In [ ]:
# análisis
import pandas as pd
import geopandas as gpd
import numpy as np

# clustering
from hdbscan import HDBSCAN

# visualización
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patheffects as path_effects
from matplotlib.colors import rgb2hex

from chiricoca.config import setup_style
setup_style(dpi=192)

In [ ]:
import sys
from pathlib import Path

EOD_PATH = Path("data") / "EOD_STGO"
EOD_PATH

In [ ]:
import huedhued.eod_scl as eod
from chiricoca.geo.utils import clip_area_geodataframe

scl_bounds = [-70.88006218, -33.67612715, -70.43015094, -33.31069169]
zones = eod.read_zone_design(EOD_PATH)
zones = clip_area_geodataframe(zones.to_crs('epsg:4326'), scl_bounds).to_crs(zones.crs)

In [ ]:
hogares = eod.read_homes(EOD_PATH)
personas = eod.read_people(EOD_PATH)

viajes = (eod.read_trips(EOD_PATH).merge(personas)).merge(hogares.drop("TipoDia", axis=1))

viajes["Peso"] = viajes["FactorExpansion"] * viajes["FactorPersona"]

In [ ]:
sns.countplot(y=viajes['Proposito'], color='grey')
sns.despine()

In [ ]:
propositos = ['Al trabajo', 'Al estudio', 'Trámites', 'De salud', 'De compras', 'Recreación']

Visualicemos los destinos de esos viajes:

In [ ]:
from chiricoca.geo.utils import to_point_geodataframe

origenes_viajes = to_point_geodataframe(
    viajes, "OrigenCoordX", "OrigenCoordY", crs="epsg:32719"
)

destinos_viajes = to_point_geodataframe(
    viajes, "DestinoCoordX", "DestinoCoordY", crs="epsg:32719"
)

In [ ]:
from chiricoca.geo.utils import clip_point_geodataframe

origenes_viajes = clip_point_geodataframe(origenes_viajes, zones.total_bounds)

destinos_viajes = destinos_viajes[
    destinos_viajes["Viaje"].isin(origenes_viajes["Viaje"])
]

destinos_viajes = clip_point_geodataframe(destinos_viajes, zones.total_bounds)

origenes_viajes = origenes_viajes[
    origenes_viajes["Viaje"].isin(destinos_viajes["Viaje"])
]

In [ ]:
from chiricoca.geo.figures import small_multiples_from_geodataframe
from chiricoca.maps import dot_map

fig, axes = small_multiples_from_geodataframe(zones, len(propositos), height=7, col_wrap=3)

for prop, ax in zip(propositos, axes.flatten()):
    zones.plot(facecolor="#efefef", edgecolor="none", ax=ax)
    dot_map(
        destinos_viajes[destinos_viajes["Proposito"] == prop],
        size=10,
        scale=0.15,
        alpha=0.5,
        ax=ax
    )
    ax.set_title(prop)

## ¿Cuántos Centros hay en la Ciudad?¿Qué los caracteriza?

Como vemos en las imágenes previas, viajes hay en toda la ciudad. Sin embargo, sabemos que no todos los sectores de la ciudad reciben afluencias grandes de personas.

El centro histórico de Santiago es sólo uno, pero lugares que concentran actividades hay varios. Entonces, ¿cuántos son? Saberlo nos permitiría apoyar la planificación de la red de transporte, el fomento de instalación en centros sub-desarrollados, o incluso la identificación de oportunidades para crear un nuevo centro.

Definiremos como _centro_ un espacio de la ciudad en el que se concentran actividades. Y, utilizando Machine Learning no supervisado, buscaremos una manera de responder la pregunta, **agrupando los destinos de los viajes como una señal de la afluencia que tiene un lugar**.

### Elección de Viajes a Considerar

No todos los viajes son iguales. Como vemos en el primer gráfico, los viajes de regreso a casa son los más frecuentes, y están dispersos por toda la ciudad. Lo que queremos son estudiar _las actividades que se hacen fuera de casa_.

Por simplicidad nos enfocaremos en las seis actividades que seleccionamos antes.

In [ ]:
analysis_trips = destinos_viajes[destinos_viajes["Proposito"].isin(propositos)].to_crs(
    "epsg:5361"
)
len(analysis_trips)

Creamos nuestra matriz de características: las posiciones de los viajes.

In [ ]:
X = np.vstack((analysis_trips.geometry.x, analysis_trips.geometry.y)).T
X

Realizamos el procedimiento estándar:

- Inicializar instancia del modelo (que ya importamos en el preámbulo) con sus hiperparámetros. Usaremos dos: 
  - `min_cluster_size`: cantidad mínima de viajes para considerar un cluster.
  - `min_samples`: cantidad mínima de puntos núcleo para considerar un cluster.
- Ajustar la matriz de características `X`
- Predecir el vector de etiquetas `y`

Los últimos dos pasos los podemos realizar en una sola llamada al método `fit_predict(X)`:

In [ ]:
dbscan = HDBSCAN(min_cluster_size=400, min_samples=25)
analysis_trips["cluster"] = dbscan.fit_predict(X)
cluster_ids = analysis_trips["cluster"].value_counts()
len(cluster_ids) - 1

Con esos hiperparámetros tenemos 9 clusters. Ésta es la cantidad de viajes que poseen:

In [ ]:
cluster_ids

Para identificar los clusters utilizaremos una paleta de colores conocida como `husl`, que varía el tono de un color a otro:

In [ ]:
colors = sns.color_palette('husl', n_colors=len(cluster_ids) - 1)
sns.palplot(colors)

Crearemos un diccionario para pintar los puntos de los clusters. Le asignaremos el color gris a los puntos de ruido:

In [ ]:
palette = dict(zip(range(len(cluster_ids) - 1), map(rgb2hex, colors)))
palette[-1] = '#afafaf'
palette

In [ ]:
cluster_centroids = {}

for cluster_id in cluster_ids.index:
    cluster_trips = analysis_trips[analysis_trips.cluster == cluster_id]

    cluster_centroids[cluster_id] = (
        cluster_trips.geometry.x.mean(),
        cluster_trips.geometry.y.mean(),
    )

cluster_centroids

In [ ]:
zones_m = zones.to_crs('epsg:5361')
fig, ax = small_multiples_from_geodataframe(zones_m, 1, height=12)

zones_m.plot(ax=ax, edgecolor="grey", facecolor="#efefef")

# pintamos los puntitos de cada cluster
for cluster_id in sorted(cluster_ids.index):
    # viajes correspondientes a cada cluster
    cluster_trips = analysis_trips[analysis_trips.cluster == cluster_id]

    # dibujamos directamente con matplotlib
    cluster_trips.plot(ax=ax, alpha=0.8, markersize=2, color=palette[cluster_id])

    # para los clusters, agregamos la etiqueta correspondiente para poder identificarlos
    # pondremos la etiqueta en el promedio de las posiciones que tiene cada cluster
    if cluster_id >= 0:
        t = ax.text(
            cluster_centroids[cluster_id][0],
            cluster_centroids[cluster_id][1],
            str(cluster_id),
            horizontalalignment="center",
            fontsize=18,
            fontweight="bold",
            color="white",
        )

        # éste es un efecto gráfico que facilita la comprensión del texto
        t.set_path_effects(
            [
                path_effects.Stroke(linewidth=2, foreground="black"),
                path_effects.Normal(),
            ]
        )


¿Qué les parecen los resultados?

Consideraciones:

- El centro histórico está correctamente identificado.
- El eje Providencia aparece como un gran centro, y parte de Vitacura y Las Condes también. Esto es coherente con los distritos comerciales y de negocios que hay en el sector.
- ¿Maipú y San Bernardo casi completas son centros? Seguramente hay que ajustar algo, porque si bien podrían serlo (sobretodo el centro de Maipú), su extensión no es tan larga.
- Plaza de Puente Alto y Concha y Toro: esto sí tiene sentido, considerando la cantidad de lugares de trabajo y servicios que hay en esos lugares.
- Vicuña Mackenna y Avenida La Florida: ídem a lo anterior.
- Gran Avenida: también, sobretodo considerando el polo comercial que es.

Hay que jugar con los hiperparámetros para entender cómo se comporta el algoritmo, y así poder ajustarlos para obtener una mejor respuesta. Verán que algunos parámetros son más sensibles que otros, y pueden introducir grandes cambios en los resultados. Al final del notebook hay algunos ejemplos sobre cómo hacerlo.

### Diversidad funcional por cluster

Calculamos la entropía de Shannon para medir la diversidad funcional de cada cluster. Un cluster con alta entropía tiene una mezcla balanceada de propósitos (multifuncional), mientras que uno con baja entropía está especializado en uno o pocos propósitos.

In [ ]:
from scipy.stats import entropy

purpose_dist = (
    analysis_trips[analysis_trips["cluster"] >= 0]
    .groupby(["cluster", "Proposito"])["Peso"]
    .sum()
    .unstack()
    .fillna(0)
    .pipe(normalize_rows)
)

purpose_dist

In [ ]:
cluster_entropy = purpose_dist.apply(entropy, axis=1)
cluster_entropy

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
cluster_entropy.sort_values().plot(kind='barh', ax=ax)
ax.set_xlabel('Entropía de Shannon')
ax.set_ylabel('Cluster')
ax.set_title('Diversidad funcional por cluster')
sns.despine()

In [ ]:
sns.heatmap(purpose_dist, annot=True, fmt='.2f', cmap='YlOrRd', linewidth=1)
plt.title('Distribución de propósitos por cluster')

Los clusters con entropía cercana a cero están especializados (por ejemplo, principalmente trabajo o comercio), mientras que aquellos con entropía más alta son centros multifuncionales donde ocurren diversos tipos de actividades.

In [ ]:
from chiricoca.geo.figures import figure_from_geodataframe
from chiricoca.colors import add_ranged_color_legend
import alphashape

fig, ax = small_multiples_from_geodataframe(zones_m, 1, height=12)

zones_m.plot(ax=ax, edgecolor="grey", facecolor="#efefef")

ent_min = cluster_entropy.min()
ent_max = cluster_entropy.max()
cmap = plt.cm.RdYlGn

for cluster_id in sorted(cluster_ids.index):
    if cluster_id < 0:
        continue
    
    cluster_trips = analysis_trips[analysis_trips.cluster == cluster_id]
    
    ent = cluster_entropy[cluster_id]
    ent_norm = (ent - ent_min) / (ent_max - ent_min)
    cluster_color = cmap(ent_norm)
    
    cluster_trips.plot(ax=ax, alpha=0.6, markersize=2, color=cluster_color)
    
    points = np.vstack((cluster_trips.geometry.x, cluster_trips.geometry.y)).T
    
    if len(points) >= 3:
        alpha_shape = alphashape.alphashape(points, alpha=0.0001)
        
        if alpha_shape and hasattr(alpha_shape, 'boundary'):
            if hasattr(alpha_shape.boundary, 'coords'):
                coords = np.array(list(alpha_shape.boundary.coords))
                ax.plot(coords[:, 0], coords[:, 1], 
                       color=cluster_color, alpha=0.9, linewidth=3)
    
    pos = cluster_centroids[cluster_id]
    t = ax.annotate(
        cluster_id,
        pos,
        horizontalalignment="center",
        va="center",
        fontsize=14,
        fontweight="bold",
        color="white",
    )
    t.set_path_effects(
        [path_effects.Stroke(linewidth=3, foreground="black"), path_effects.Normal()]
    )

n_bins = 5
bins = np.linspace(ent_min, ent_max, n_bins + 1)
palette = [rgb2hex(cmap(i / n_bins)) for i in range(n_bins)]

add_ranged_color_legend(
    ax,
    bins=bins,
    built_palette=palette,
    location="lower left",
    orientation="horizontal",
    label="Entropía (especializado a multifuncional)",
    width="30%",
    height="3%",
    bbox_to_anchor=(0.02, 0.02, 1, 1)
)

### ¿Qué más hacer?

Otra opción es tratar de buscar maneras de evaluar los resultados. Por ejemplo, comparando los clusters obtenidos con el uso de suelo del Servicio de Impuestos Internos, o los planes reguladores de cada comuna. ¿Coinciden los bordes obtenidos con los definidos por las autoridades? ¿O el uso de la ciudad se desmarca de los bordes impuestos administrativamente?

Una manera podría ser visualizar dónde vive la gente que trabaja en cada cluster:

In [ ]:
origenes_viajes = to_point_geodataframe(
    analysis_trips[analysis_trips['cluster'] >= 0], "OrigenCoordX", "OrigenCoordY", "epsg:5361"
)

In [ ]:
fig, axes = small_multiples_from_geodataframe(
    zones_m, len(cluster_ids) - 1, height=7, col_wrap=4
)

for cluster_id, ax in zip(range(cluster_ids.index.max() + 1), axes.flatten()):
    if cluster_id < 0:
        continue
    zones_m.plot(facecolor="#efefef", edgecolor="none", ax=ax)
    origenes_viajes[origenes_viajes["cluster"] == cluster_id].plot(ax=ax, markersize=2, marker='.')
    analysis_trips[analysis_trips['cluster'] == cluster_id].plot(ax=ax, color=palette[cluster_id], markersize=2, marker='.')
    ax.set_title(cluster_id)

## Ajuste de hiperparámetros

Los resultados de HDBSCAN dependen de tres hiperparámetros principales:

- `min_cluster_size`: tamaño mínimo para formar un cluster (clusters pequeños se consideran ruido)
- `min_samples`: número de vecinos que debe tener un punto para ser considerado núcleo de un cluster (valores bajos = clusters más flexibles, valores altos = clusters más densos y conservadores)
- `cluster_selection_epsilon`: distancia mínima entre clusters (ayuda a fusionar clusters cercanos)

Exploremos cómo afectan los resultados.

In [ ]:
from sklearn.metrics import silhouette_score

def evaluate_clustering(X, min_cluster_size):
    min_samples = int(np.log(X.shape[0]))
    epsilon = 10
    hdb = HDBSCAN(
        min_cluster_size=min_cluster_size, 
        min_samples=min_samples,
        cluster_selection_epsilon=epsilon
    )
    labels = hdb.fit_predict(X)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    pct_noise = 100 * n_noise / len(labels)
    
    # Silhouette score (solo para puntos no-ruido)
    if n_clusters > 1:
        mask = labels != -1
        if mask.sum() > 0:
            score = silhouette_score(X[mask], labels[mask])
        else:
            score = -1
    else:
        score = -1
    
    return {
        'n_clusters': n_clusters,
        'pct_noise': pct_noise,
        'silhouette': score,
        'min_cluster_size': min_cluster_size,
        'min_samples': min_samples,
        'cluster_selection_epsilon': epsilon,
        'labels': labels
    }

configs = [100, 200, 400, 600]

results = [evaluate_clustering(X, c) for c in configs]
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'labels'} for r in results])
results_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

results_df.plot(x='min_cluster_size', y='n_clusters', ax=axes[0], marker='o', legend=False)
axes[0].set_ylabel('Número de clusters')
axes[0].set_title('Clusters generados')

results_df.plot(x='min_cluster_size', y='pct_noise', ax=axes[1], marker='o', legend=False, color='orange')
axes[1].set_ylabel('% puntos como ruido')
axes[1].set_title('Ruido generado')

results_df.plot(x='min_cluster_size', y='silhouette', ax=axes[2], marker='o', legend=False, color='green')
axes[2].set_ylabel('Silhouette score')
axes[2].set_title('Calidad de clusters')

plt.tight_layout()
sns.despine()

In [ ]:
fig, axes = small_multiples_from_geodataframe(zones_m, len(results), height=10, col_wrap=2)

for res, ax in zip(results, axes.flatten()):
    zones_m.plot(ax=ax, edgecolor="grey", facecolor="#efefef")
    
    temp_trips = analysis_trips.copy()
    temp_trips['cluster'] = res['labels']
    
    noise = temp_trips[temp_trips['cluster'] == -1]
    clusters = temp_trips[temp_trips['cluster'] >= 0]
    
    if len(noise) > 0:
        noise.plot(ax=ax, color='grey', alpha=0.3, markersize=1)
    
    if len(clusters) > 0:
        clusters.plot(ax=ax, column='cluster', cmap='tab10', alpha=0.6, markersize=2, legend=False)
    
    ax.set_title(f"min_size={res['min_cluster_size']}, min_samples={res['min_samples']}\n{res['n_clusters']} clusters, {res['pct_noise']:.1f}% ruido")